# Why type hints?

Python is dynamically typed: variables don't have declared types, and the interpreter figures out what operations are valid at runtime. This is flexible, but it means the program can get some way through an execution before something explodes.

Type *hints* — the annotations you'll see in modern Python code — describe the types you intend things to have. Python itself doesn't use them for anything at runtime. A separate tool (`mypy`, `pyright`, or similar) reads them and flags mismatches *before* you run the code. Your editor uses them too, for autocomplete and inline warnings.

This notebook covers what type hints are, what they aren't, and the problem they solve.

## What a type hint looks like

A function with no hints:

In [ ]:
def greet(name):
    return f"Hello, {name}"

print(greet("Matthew"))

The same function with type hints — `name: str` means "`name` should be a string" and `-> str` means "the return value is a string":

In [ ]:
def greet(name: str) -> str:
    return f"Hello, {name}"

print(greet("Matthew"))

The behaviour at runtime is *identical* — Python runs both functions exactly the same way. The only difference is the annotations, which are there for humans and tooling to read.

## Python doesn't enforce type hints

This is worth being clear about. Annotations are just metadata — Python stores them on the function object but doesn't check anything. You can pass the "wrong" type and Python will run the function anyway, raising an error (or not) only if the function body actually tries something the type doesn't support:

In [ ]:
def greet(name: str) -> str:
    return f"Hello, {name}"

# Passing an int — Python doesn't care
print(greet(42))

# The annotations are still stored, though
print(greet.__annotations__)

That `42` ran just fine because f-strings happily accept any type. The annotation said "this should be a string"; Python didn't check; the code still worked. This is central to how Python type hints are designed — they're *declarative*, not *enforcing*.

## Where the value comes from

Two audiences make annotations useful:

**A type-checker** (mypy, pyright) reads annotations across your whole codebase and reports mismatches — "you called `greet(42)` but `greet` expects a `str`". Run this in CI and bugs get caught before the code ever executes.

**Your editor** uses them for autocomplete ("what methods can I call on this?") and for inline warnings ("this call won't type-check"). VS Code's Pylance, PyCharm's inspector, Zed's LSP — they all do this. You see the benefit without running anything.

A function with good type hints is also self-documenting — the reader doesn't have to trace through the body to learn what shape the inputs and outputs are.

## Running `mypy`

You can't run mypy in a Jupyter cell easily, but it's worth seeing what it catches. Given this function in a file `greet.py`:

In [ ]:
# Contents of greet.py:
source = '''
def greet(name: str) -> str:
    return f"Hello, {name}"

greet(42)          # should be str, not int
'''

# Running `mypy greet.py` would produce:
print('greet.py:5: error: Argument 1 to "greet" has incompatible type "int"; expected "str"')
print('Found 1 error in 1 file (checked 1 source file)')

No code has been executed. `mypy` analyses the source, sees the mismatch, and reports it. That's the whole loop: write code with annotations, run a checker, fix what it flags.

Most teams wire this into pre-commit or CI so the checks happen automatically.

## What problems this actually solves

Type hints are most useful when they catch bugs that would otherwise surface as confusing runtime errors far from the original mistake. Canonical examples:

In [ ]:
# Imagine a real codebase — functions call each other, data flows around.
# A function signature change in one place can break callers in many places.

def parse_user_id(raw: str) -> int:
    return int(raw.strip())

def fetch_user(user_id: int) -> dict:
    # ... imagine a database lookup ...
    return {"id": user_id, "name": "Alice"}

# If someone refactors parse_user_id to return a str, every caller of fetch_user
# that uses its output is now broken. A type-checker flags this immediately.
# Without type-checking, you'd find out when a test (or worse, production) failed.

raw = "42"
uid = parse_user_id(raw)
user = fetch_user(uid)
print(user)

The bugs that get caught tend to be:

- Passing the wrong type to a function (often the most useful category).
- Forgetting to handle `None` returns.
- Misremembering the shape of a data structure ("is this a `list[dict]` or a `dict[str, list]`?").
- Refactoring broken contracts — changing one function's signature and missing the callers that depended on it.

Things type hints *don't* catch: logic errors, off-by-ones, using the wrong algorithm. A type-checker verifies the shapes fit, not that the code does what it should.

## The cost

Annotations take time to write. They can clutter simple code (`def add(a: int, b: int) -> int: return a + b`). They sometimes get awkward — typing a function that accepts "anything with a `.read()` method" needs a `Protocol`, which is more typing than the function itself.

For short scripts or one-off notebooks, the cost-benefit is often not there. For libraries, long-lived applications, and anything that multiple people will touch, type hints pay for themselves quickly — the [when type hints help essay](../concepts/when-type-hints-help.md) has the full argument.

## Gradual typing

Python's type system is *gradual*: you don't have to type everything. Annotate the parts you care about, leave the rest alone, and the type-checker silently treats untyped code as `Any` (everything is permitted, nothing is checked).

This is a real feature. It means you can add types to a legacy codebase one module at a time, or leave your test files unannotated while still typing the production code carefully. See the [gradual typing essay](../concepts/gradual-typing.md) for what this buys you.

## Recap

- Type hints are annotations — they describe intended types but don't affect runtime behaviour.
- A *type-checker* (mypy, pyright) reads them and flags mismatches before execution.
- Editors use them for autocomplete and inline warnings.
- They catch type-shape bugs, not logic bugs. Worth the cost on long-lived code; skip for one-off scripts.
- Gradual: annotate some things, not others — a legit workflow.

Next: [Basic annotations](02-basic-annotations.ipynb), the syntax for typing variables, parameters, and return values.